In [1]:
from pymongo import MongoClient
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
print(sys.executable)

c:\Users\gayat\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
client = MongoClient("mongodb://localhost:27017/")

db = client["MovieRecommendationDB"]

collection = db["recommendations"]

print("Connected Successfully!")

Connected Successfully!


In [3]:
with open("../svd_model.pkl", "rb") as f:
    model = pickle.load(f)

U = model["U"]
sigma = model["sigma"]
Vt = model["Vt"]
user_map = model["user_map"]
movie_map = model["movie_map"]

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [4]:
movies = pd.read_csv("../ml-25m/movies.csv")

movie_ids = list(movie_map.keys())

documents = []

for user in range(100):      # First 100 users only

    scores = np.dot(np.dot(U[user], sigma), Vt)

    top100 = np.argsort(scores)[::-1][:100]

    recommended_ids = [movie_ids[i] for i in top100]

    recs = []

    for movie_id in recommended_ids:

        movie = movies[movies["movieId"] == movie_id].iloc[0]

        recs.append({
            "movieId": int(movie_id),
            "title": movie["title"],
            "genres": movie["genres"]
        })

    documents.append({
        "userId": int(user),
        "recommendations": recs
    })

print("Prepared", len(documents), "documents")

Prepared 100 documents


In [5]:
db = client["MovieLensDB"]

collection = db["recommendations"]

collection.delete_many({})

collection.insert_many(documents)

print("Recommendations stored successfully!")

Recommendations stored successfully!


In [6]:
print(collection.count_documents({}))

100


In [7]:
sample = collection.find_one({"userId": 0})

print(sample)

{'_id': ObjectId('6a4bf1ebd5ad20f752585d87'), 'userId': 0, 'recommendations': [{'movieId': 364, 'title': 'Lion King, The (1994)', 'genres': 'Adventure|Animation|Children|Drama|Musical|IMAX'}, {'movieId': 1210, 'title': 'Star Wars: Episode VI - Return of the Jedi (1983)', 'genres': 'Action|Adventure|Sci-Fi'}, {'movieId': 3578, 'title': 'Gladiator (2000)', 'genres': 'Action|Adventure|Drama'}, {'movieId': 1193, 'title': "One Flew Over the Cuckoo's Nest (1975)", 'genres': 'Drama'}, {'movieId': 5952, 'title': 'Lord of the Rings: The Two Towers, The (2002)', 'genres': 'Adventure|Fantasy'}, {'movieId': 595, 'title': 'Beauty and the Beast (1991)', 'genres': 'Animation|Children|Fantasy|Musical|Romance|IMAX'}, {'movieId': 480, 'title': 'Jurassic Park (1993)', 'genres': 'Action|Adventure|Sci-Fi|Thriller'}, {'movieId': 593, 'title': 'Silence of the Lambs, The (1991)', 'genres': 'Crime|Horror|Thriller'}, {'movieId': 2571, 'title': 'Matrix, The (1999)', 'genres': 'Action|Sci-Fi|Thriller'}, {'movieId